<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_DLinear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
#!git clone "https://github.com/slomi23/ML_fx.git"
#!cd ML_fx/

# Fetch Data

In [41]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

from google.colab import drive

drive.mount("/content/drive")

train = pd.read_csv("/content/drive/MyDrive/train_prepared.csv")
print(train.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0        0.0        0.0        0.0  ...    20  151315           NaN  2010   
1        0.0        0.0        0.0  ...    20  151315           NaN  2010   
2        0.0        0.0        0.0  ...    20  151315           NaN  2010   
3        0.0        0.0        0.0  ...    20  151315           NaN  2010   
4    

# Preparing for training and validation

In [42]:
split_date = '2011-12-01'

# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")

Final Training Set Shape: (279085, 24)
Validation Set Shape: (142485, 24)
Validation Period: 2011-12-02 to 2012-10-26


# W&B

In [43]:
!pip install wandb -q
!pip install neuralforecast pytorch-lightning wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


# Windows(52 weeks)

In [44]:
import numpy as np
import pandas as pd
import torch

LOOKBACK = 52
HORIZON = 39
VAL_START_DATE = '2011-12-01'

def build_windows(df, lookback=52, horizon=39, val_start_date='2011-12-01'):
    df = df.sort_values(['Store', 'Dept', 'Date']).copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df['IsHoliday'] = df['IsHoliday'].astype(int) if 'IsHoliday' in df.columns else 0

    train_X, train_y, train_holiday = [], [], []
    val_X, val_y, val_holiday, val_meta = [], [], [], []

    for (store, dept), g in df.groupby(['Store', 'Dept']):
        g = g.reset_index(drop=True)
        sales = g['Weekly_Sales'].values
        dates = g['Date'].values
        holidays = g['IsHoliday'].values

        if len(g) < (lookback + horizon):
            continue

        for t in range(lookback, len(g) - horizon + 1):
            window = sales[t - lookback:t]
            target = sales[t:t + horizon]
            target_date = dates[t]
            target_holidays = holidays[t:t + horizon]

            if target_date < np.datetime64(val_start_date):
                train_X.append(window)
                train_y.append(target)
                train_holiday.append(target_holidays)
            else:
                val_X.append(window)
                val_y.append(target)
                val_holiday.append(target_holidays)
                val_meta.append((store, dept, target_date))

    return (
        np.array(train_X, dtype=np.float32), np.array(train_y, dtype=np.float32), np.array(train_holiday, dtype=np.float32),
        np.array(val_X, dtype=np.float32), np.array(val_y, dtype=np.float32), np.array(val_holiday, dtype=np.float32), val_meta
    )

df_combined = pd.concat([train_set, val_set]).reset_index(drop=True)

train_X, train_y, train_holiday, val_X, val_y, val_holiday, val_meta = build_windows(
    df_combined, LOOKBACK, HORIZON, VAL_START_DATE
)

print(f"Train windows: {len(train_X)} | Val windows: {len(val_X)}")
print(f"Target shape per window: {train_y.shape[1]} weeks")

Train windows: 121140 | Val windows: 27929
Target shape per window: 39 weeks


# Window Normalization

In [45]:
from torch.utils.data import Dataset, DataLoader

class SalesWindowDataset(Dataset):
    def __init__(self, X, y, holidays):
        self.X = X
        self.y = y
        self.holidays = holidays
        # Shape: (N, 1) to enable proper broadcasting
        self.scale = np.clip(np.abs(X).mean(axis=1, keepdims=True), 1.0, None)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_scaled = self.X[idx] / self.scale[idx]
        y_scaled = self.y[idx] / self.scale[idx]

        return (
            torch.tensor(x_scaled, dtype=torch.float32),
            torch.tensor(y_scaled, dtype=torch.float32),
            torch.tensor(self.scale[idx], dtype=torch.float32), # Shape (1,)
            torch.tensor(self.holidays[idx], dtype=torch.float32)
        )

BATCH_SIZE = 256
train_ds = SalesWindowDataset(train_X, train_y, train_holiday)
val_ds = SalesWindowDataset(val_X, val_y, val_holiday)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# კომპონენტებად დაყოფა

In [46]:
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        front = x[:, 0:1].repeat(1, (self.kernel_size - 1) // 2)
        end = x[:, -1:].repeat(1, (self.kernel_size - 1) // 2)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.unsqueeze(1)).squeeze(1)

class DLinear(nn.Module):
    def __init__(self, lookback=52, horizon=39, kernel_size=25):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        out = self.linear_trend(trend) + self.linear_seasonal(seasonal)
        return out

model = DLinear(lookback=LOOKBACK, horizon=HORIZON, kernel_size=25).to(DEVICE)
print(model)

Using device: cuda
DLinear(
  (moving_avg): MovingAvg(
    (avg): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(0,))
  )
  (linear_trend): Linear(in_features=52, out_features=39, bias=True)
  (linear_seasonal): Linear(in_features=52, out_features=39, bias=True)
)


# Training

In [47]:
import os
import torch.optim as optim
import wandb

EPOCHS = 30
LR = 1e-3

run = wandb.init(
    project="ML_fx_DLinear_PyTorch",
    name="DLinear_39_Horizon_Fixed",
    config={
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "kernel_size": 25,
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": 42,
    },
)

optimizer = optim.Adam(model.parameters(), lr=LR)
best_val_wmae = float("inf")
best_val_mae = float("inf")

print("Starting custom PyTorch training loop...")

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum, train_mae_sum, train_wmae_sum = 0, 0, 0
    train_steps, train_total_weights = 0, 0

    for x, y, scale, holiday in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        scale, holiday = scale.to(DEVICE), holiday.to(DEVICE)

        optimizer.zero_grad()
        preds = model(x)  # Shape: [batch_size, 39]
        weights = torch.where(holiday == 1.0, 5.0, 1.0)

        preds_real = preds * scale  # Shape: [batch_size, 39]
        y_real = y * scale          # Shape: [batch_size, 39]

        loss = torch.sum(weights * torch.abs(preds_real - y_real)) / torch.sum(weights)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_mae = torch.sum(torch.abs(preds_real - y_real))
        batch_wmae = torch.sum(weights * torch.abs(preds_real - y_real))

        train_loss_sum += loss.item()
        train_mae_sum += batch_mae.item()
        train_wmae_sum += batch_wmae.item()
        train_total_weights += torch.sum(weights).item()
        train_steps += y.numel()

    avg_train_loss = train_loss_sum / len(train_loader)
    avg_train_mae = train_mae_sum / train_steps
    avg_train_wmae = train_wmae_sum / train_total_weights

    model.eval()
    val_mae_sum, val_wmae_sum = 0, 0
    val_steps, val_total_weights = 0, 0

    with torch.no_grad():
        for x, y, scale, holiday in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            scale, holiday = scale.to(DEVICE), holiday.to(DEVICE)

            preds = model(x)
            weights = torch.where(holiday == 1.0, 5.0, 1.0)
            preds_real = preds * scale
            y_real = y * scale

            batch_mae = torch.sum(torch.abs(preds_real - y_real))
            batch_wmae = torch.sum(weights * torch.abs(preds_real - y_real))

            val_mae_sum += batch_mae.item()
            val_wmae_sum += batch_wmae.item()
            val_total_weights += torch.sum(weights).item()
            val_steps += y.numel()

    avg_val_mae = val_mae_sum / val_steps
    avg_val_wmae = val_wmae_sum / val_total_weights

    if avg_val_wmae < best_val_wmae:
        best_val_wmae = avg_val_wmae
        best_val_mae = avg_val_mae
        torch.save(model.state_dict(), "best_dlinear_model.pth")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_mae": avg_train_mae,
        "train_wmae": avg_train_wmae,
        "val_mae": avg_val_mae,
        "val_wmae": avg_val_wmae,
    })

print("-" * 50)
print(f"Final Best Validation MAE:  {best_val_mae:.2f}")
print(f"Final Best Validation WMAE: {best_val_wmae:.2f}")
print("-" * 50)

artifact = wandb.Artifact("custom-dlinear-model", type="model")
artifact.add_file("best_dlinear_model.pth")
run.log_artifact(artifact)

run.link_artifact(
    artifact,
    target_path="wandb-registry-model/Walmart-DLinear",
    aliases=["latest"],
)

wandb.finish()

Starting custom PyTorch training loop...
--------------------------------------------------
Final Best Validation MAE:  2053.23
Final Best Validation WMAE: 2073.88
--------------------------------------------------


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_mae,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_wmae,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,█▃▂▄▂▂▃▂▃▂▂▃▁▂▂▂▃▁▁▂▂▂▃▃▂▂▂▁▂▃
val_wmae,█▃▂▄▂▂▂▂▄▂▂▃▁▃▂▃▃▁▂▂▂▂▃▃▂▃▂▂▄▃
epoch,30
train_loss,1896.61467
train_mae,1853.56941
train_wmae,1896.4198
val_mae,2087.35829
